# QuantiPhy · IIIXR 2D Static (`SS` + `DS`) · Sanity (6 câu / 3 video trên GPU T4 ×2)

Kiểm tra toàn diện đường đi của nhánh **đo kích thước & khoảng cách 2D tĩnh (`2d_ratio` / `sruler` / `vlm_2d_ratio`)** từ raw video + câu hỏi QuantiPhy validation tới đại lượng vật lý `fresh_box` trên 6 câu mốc thuộc 3 video ngắn:
- `#2151`, `#2152` (`internet_0001`, `SS` · `size`: chiều dài & chiều rộng ô tô chuẩn hóa bằng bề rộng làn đường từ `scene_ruler`)
- `#2272`, `#2275` (`internet_0027`, `SS` · `dist2`: khoảng cách giữa 2 quả bóng màu tại `when="start"` và `when="end"`, kiểm tra phân biệt màu `colour_assign` + khoảng cách đa giác `_poly_dist`)
- `#402`, `#403` (`simulation_0163`, `SS` · `size`: `#402` chiều cao bowling pin qua `vlm_2d_ratio` `v3nt` fallback và `#403` kiểm tra chốt chặn `target=prior_object` chủ động abstain)

Đồng thời chạy phép thử đối chứng **bật/tắt trace** cùng `seed=0` trên `#2272` (`internet_0027`), kiểm chứng tái tính `fresh_box` thuần CPU từ mảng `.npy` đã lưu (`run/verify.py`), rồi đóng gói `/kaggle/working/iiixr_2d_static_sanity.tar` kèm trang phân tích lỗi offline `run/index.html`.

**Thiết lập Kaggle:** Chọn Accelerator **GPU T4 ×2** và bật **Internet** (hoặc gắn snapshot dataset/model + `quantiphy_full.tar` qua **Add Input** khi chạy offline).

## 1. Cấu hình

In [ ]:
REPO_URL = "https://github.com/leetranwyng/quantiphy_trannl.git"
SOURCE_REF = "main"  # Có thể pin trực tiếp commit SHA cụ thể
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # Chỉ dùng nếu repo chuyển sang Private

# Để trống để tự tải snapshot đã pin từ HuggingFace; hoặc điền đường dẫn trong /kaggle/input khi chạy offline.
DATASET_DIR = ""
MODELS_DIR = ""
CAW_FULL_TAR = ""  # Để trống để tự tìm quantiphy_full.tar trong /kaggle/input nếu đã gắn
WHEELHOUSE = ""

SCOPE = "static2d_sanity"   # 6 câu 2D tĩnh trên 3 video: internet_0001, internet_0027, simulation_0163
DTYPE_PROFILE = "t4_fp16"   # "t4_fp16" (tối ưu T4 SM 7.5), "fp32", hoặc "upstream_bf16"
RULE_PARSE_ONLY = False     # False = chạy cả Qwen3-VL-8B cho bước parse fallback + vlm_ground_notext (cần cho #402)
OUTPUT_BUDGET_GIB = 18.0

## 2. Clone repo tại ref đã chọn & cài đặt dependencies đã khóa

In [ ]:
import base64, os, subprocess, sys, tempfile
from pathlib import Path

source = Path(tempfile.mkdtemp(prefix="iiixr-st2d-")) / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
clone = ["git", "-c", "credential.helper=", "clone", "--quiet", REPO_URL, str(source)]
if subprocess.run(clone, env=env).returncode:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    subprocess.run(clone, check=True, env=dict(env, GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraHeader",
        GIT_CONFIG_VALUE_0="Authorization: Basic " + auth))
    del token, auth

remote_ref = f"refs/remotes/origin/{SOURCE_REF}"
is_branch = subprocess.run(["git", "show-ref", "--verify", "--quiet", remote_ref], cwd=source).returncode == 0
subprocess.run(["git", "checkout", "--quiet", "--detach", remote_ref if is_branch else SOURCE_REF], cwd=source, check=True)
print("Resolved source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source, text=True).strip())

install = [sys.executable, "-m", "pip", "install", "-q", "-r", str(source / "solutions/iiixr/kaggle/requirements.txt")]
subprocess.run(install + (["--no-index", "--find-links", WHEELHOUSE] if WHEELHOUSE else []), check=True)


## 3. Thực thi Sanity 2D Static & xuất archive + viewer offline

In [ ]:
import runpy
sys.path.insert(0, str(source))
launch = runpy.run_path(str(source / "solutions/iiixr/kaggle/notebook.py"))["launch"]
SUMMARY = launch(source, "sanity", globals())
